# Predicción de retrasos en vuelos comerciales
Grupo 11: Mikel Lorite, Adrián Izquierdo y Jon Alba

## Contexto y objetivos del proyecto
El contexto de este proyecto se enmarca en la industria de la aviación comercial, un sector que genera cantidades masivas de datos diariamente y donde la puntualidad operativa es un factor crítico. Para llevar a cabo este estudio, utilizamos el conjunto de datos "2015 Flight Delays and Cancellations", publicado originalmente por la Oficina de Estadísticas de Transporte (Bureau of Transportation Statistics) del Departamento de Transporte de los Estados Unidos (DOT). Este dataset rastrea el rendimiento y la puntualidad de los vuelos domésticos operados por las grandes aerolíneas comerciales dentro del país durante el año 2015.

El objetivo principal de nuestro trabajo es construir un modelo predictivo escalable utilizando Apache Spark que permita anticipar si un vuelo sufrirá un retraso significativo a su llegada. A través de este análisis, buscamos identificar patrones subyacentes en las demoras, respondiendo a preguntas sobre qué factores —como la aerolínea, las infraestructuras de origen/destino o las franjas horarias— inciden en la probabilidad de que un vuelo no cumpla con su horario programado. Adicionalmente, desde la perspectiva de la Ingeniería de Datos, el proyecto tiene como meta evaluar la eficiencia computacional de diversos algoritmos de clasificación binaria (Regresión Logística, Random Forest y Gradient-Boosted Trees) frente a un escenario de alto volumen de datos, midiendo y documentando empíricamente la escalabilidad del clúster a través de pruebas de speed-up y size-up.

## Descripción de los datos 
El conjunto de datos seleccionado representa un volumen de información de gran magnitud, ideal para el procesamiento distribuido. En su totalidad, consta de más de 5,8 millones de registros (5.819.079 observaciones empíricas) estructurados originalmente en 31 variables. Para garantizar la integridad relacional de la información, el dataset se divide en tres archivos en formato CSV independientes pero interconectados:

* flights.csv: Constituye el núcleo central de la información, recogiendo el registro individual de cada vuelo operado durante el año. Entre sus variables métricas y categóricas se incluye información temporal exhaustiva (fecha, horarios de salida y llegada programados frente a los reales) e indicadores de rendimiento operativo. La variable clave para nuestro problema de clasificación subyace en la columna de retraso a la llegada (Arrival Delay), que nos indica en minutos la diferencia respecto al horario previsto.

* airlines.csv: Funciona como una tabla de dimensión o diccionario. Contiene el mapeo directo entre los identificadores de código IATA y el nombre comercial estandarizado de cada compañía aérea, lo que resulta fundamental para la posterior visualización e interpretación de los modelos.

* airports.csv: Actúa como un catálogo geográfico detallado. Este archivo permite enriquecer los datos de los vuelos enlazando los códigos IATA de los aeropuertos de origen y destino con su ubicación física. Proporciona variables demográficas y geoespaciales clave como la ciudad, el estado, la latitud y la longitud, abriendo la puerta a capturar el impacto de la congestión regional en nuestras predicciones.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os


spark = SparkSession.builder.appName("FlightDelays").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 19:10:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import pyspark.sql.functions as F

df_flights = spark.read.csv("data/flights.csv", header=True, inferSchema=True)
df_airlines = spark.read.csv("data/airlines.csv", header=True, inferSchema=True)
df_airports = spark.read.csv("data/airports.csv", header=True, inferSchema=True)

print(f"Total de vuelos iniciales: {df_flights.count():,}")

Total de vuelos iniciales: 5,819,079


In [5]:
df_joined = df_flights.join(
    F.broadcast(df_airlines), 
    df_flights.AIRLINE == df_airlines.IATA_CODE, 
    "left"
).drop("IATA_CODE") 

df_airports_orig = df_airports.select(
    F.col("IATA_CODE").alias("ORIGIN_IATA"),
    F.col("LATITUDE").alias("ORIGIN_LAT"),
    F.col("LONGITUDE").alias("ORIGIN_LONG")
)

df_joined = df_joined.join(
    F.broadcast(df_airports_orig), 
    df_joined.ORIGIN_AIRPORT == df_airports_orig.ORIGIN_IATA, 
    "left"
).drop("ORIGIN_IATA")

# Puedes repetir este último paso para DESTINATION_AIRPORT si lo consideras útil

In [9]:
# Filtrar cancelados y desviados
df_clean = df_joined.filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))

# Contar nulos en las columnas clave
df_clean.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in ["ARRIVAL_DELAY", "DEPARTURE_DELAY"]]).show()

+-------------+---------------+
|ARRIVAL_DELAY|DEPARTURE_DELAY|
+-------------+---------------+
|            0|              0|
+-------------+---------------+



Tras la ejecución del proceso de limpieza, los reusltado obtenidos confirman la viabilidad del dataset para el entrenamiento de modelos.
El filtrado de vuelos cancelados y desviados ha eliminado los valores faltantes. Como se observa en la salida anterior, ambas variables críticas para la predicción (ARRIVAL_DELAY y DEPARTURE_DELAY) presentan 0 valores nulos. Al disponer de un conjunto de datos limpio con más de 5 millones de registros, garantizamos que las métricas size-up y speed-up que analizaremos más adelante sean representativas del redimiento real.  

In [10]:
df_clean.select("ARRIVAL_DELAY", "DEPARTURE_DELAY", "DISTANCE").summary().show()

26/05/04 19:38:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+-----------------+-----------------+
|summary|    ARRIVAL_DELAY|  DEPARTURE_DELAY|         DISTANCE|
+-------+-----------------+-----------------+-----------------+
|  count|          5714008|          5714008|          5714008|
|   mean|4.407057357987598| 9.29484190431655|824.4569032804994|
| stddev|39.27129709388608|36.88972372075696|608.6619895866803|
|    min|              -87|              -82|               31|
|    25%|              -13|               -5|              373|
|    50%|               -5|               -2|              650|
|    75%|                8|                7|             1065|
|    max|             1971|             1988|             4983|
+-------+-----------------+-----------------+-----------------+



Tras obtener el resumen estadístico de las variables clave podemos extraer las siguientes conclusiones:
- La mediana de ARRIVAL_DELAY es de -5 minutos, lo que significa que más de la mitad de los vuelos llegan antes de su horario programado. Sin embargo, la media es de 4.4 minutos. Esta discrepancia confirma una asimetría positiva en la distribución, causada por una "cola larga" de vuelos con retrasos significativos.
- La desciación típica de los retrasos es casi diez veces superior a la media. Esto indica que mientras que la mayoría de los vuelos son puntuales, los casos de retraso son bastante extremos.
- Se observan valores máximos de hasta 1971 minutos (aproximadamente 32h). Estos outliers son críticos, representan casis que el modelo debe aprender a diferenciar de las fluctuaciones normales de 10 o 15 minutos.
- Existe una diferencia de magnitudes masiva entre las variables. Mientras que los retrasos se mueven en un rango de decenas, las variable DISTANCE alcanza valores de 4983 millas. Esto es lo que justifica la necesidad de aplicar una normalización, evitando que la distancia domine artificialmente los cálculos.
- Dado que el percentil 75% de los retrasos se sitúa en 8 minutos, usaremos el estándar de la FAA (Federal Aviation Administration o Administración Federal de Aviación) que son a partir de los 15 minutos, cuando se considera que un vuelo está retrasado, ya que es cuando empieza a causar problemas como por ejemplo: los pasajeros pierdan sus conexiones, la tripulación puede exceder sus horas legales de trabajo...  Esto nos permitirá centrar la capacidad predictiva del sistema en los retrasos que realmente generan un impacto logístico y económico negativo.

De acuerdo a nuestro objetivo del trabajo, debemos crear una variable que nos especifique si el vuelo se ha retrasado o no, ya que nuestras dos variables binarias del csv nos reflejan si están desviados o calcelados. Para ello usaremos el criterio que hemos mencionado antes.

In [12]:
df_model = df_clean.withColumn("LABEL", F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0))

df_model.groupBy("LABEL").count().show()

+-----+-------+
|LABEL|  count|
+-----+-------+
|    1|1063439|
|    0|4650569|
+-----+-------+



El dataser presenta un desbalanceo de clases moderado proporcionando un 18.6% para los vuelos con retraso y un 81.4% para los vuelos puntuales. El volumen de casos positivos es lo suficiente como para que los algoritmos de Spark aprendan los aptrones del retraso sin necesidad de aplicar técnicaas de sobremuestreo (oversampling).

In [17]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

# 1. Seleccionamos las variables numéricas candidatas (excluyendo "causas" y IDs)
numeric_cols = [
    "YEAR", "MONTH", "DAY" ,"DAY_OF_WEEK", "FLIGHT_NUMBER","SCHEDULED_DEPARTURE", "DEPARTURE_TIME", 
    "DEPARTURE_DELAY", "TAXI_OUT", "WHEELS_OFF", "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", 
    "DISTANCE", "WHEELS_ON", "TAXI_IN", "SCHEDULED_ARRIVAL", "ARRIVAL_TIME", "ARRIVAL_DELAY", "DIVERTED", "CANCELLED"
]

# 2. Preparamos el vector (Spark requiere este paso previo)
assembler = VectorAssembler(inputCols=numeric_cols, outputCol="corr_features")
df_vector = assembler.transform(df_clean).select("corr_features")

# 3. Calculamos la matriz de Pearson
matrix = Correlation.corr(df_vector, "corr_features").collect()[0][0]
cor_np = matrix.toArray()

# 4. Lo pasamos a un DataFrame de Pandas para que se vea bonito en el informe
import pandas as pd
corr_matrix_df = pd.DataFrame(cor_np, columns=numeric_cols, index=numeric_cols)

# Mostramos la matriz redondeada a 2 decimales para que sea legible
print("--- Matriz de Correlación Completa ---")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(corr_matrix_df.round(2))

--- Matriz de Correlación Completa ---


26/05/04 20:47:03 WARN PearsonCorrelation: Pearson correlation matrix contains NaN values.


,YEAR,MONTH,DAY,DAY_OF_WEEK,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED
YEAR,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MONTH,NaN,1.00,0.01,-0.01,-0.02,-0.00,-0.00,-0.02,-0.01,-0.00,0.01,0.00,0.00,0.01,-0.01,0.00,-0.01,-0.01,-0.04,NaN,NaN
DAY,NaN,0.01,1.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,NaN,NaN
DAY_OF_WEEK,NaN,-0.01,0.00,1.00,0.02,0.01,0.01,-0.01,-0.02,0.00,0.01,0.01,0.01,0.02,0.01,0.00,0.01,0.01,-0.02,NaN,NaN
FLIGHT_NUMBER,NaN,-0.02,0.00,0.02,1.00,-0.01,-0.00,-0.01,0.05,0.01,-0.32,-0.31,-0.32,-0.33,-0.01,-0.02,-0.02,-0.00,0.02,NaN,NaN
SCHEDULED_DEPARTURE,NaN,-0.00,-0.00,0.01,-0.01,1.00,0.96,0.11,0.01,0.94,-0.02,-0.02,-0.02,-0.01,0.66,-0.04,0.71,0.63,0.10,NaN,NaN
DEPARTURE_TIME,NaN,-0.00,-0.00,0.01,-0.00,0.96,1.00,0.17,0.01,0.97,-0.02,-0.02,-0.02,-0.02,0.68,-0.04,0.71,0.65,0.16,NaN,NaN
DEPARTURE_DELAY,NaN,-0.02,-0.00,-0.01,-0.01,0.11,0.17,1.00,0.06,0.16,0.03,0.03,0.02,0.02,0.06,0.01,0.10,0.05,0.94,NaN,NaN
TAXI_OUT,NaN,-0.01,-0.00,-0.02,0.05,0.01,0.01,0.06,1.00,0.04,0.11,0.21,0.09,0.07,0.03,0.00,0.02,0.03,0.23,NaN,NaN
WHEELS_OFF,NaN,-0.00,-0.00,0.00,0.01,0.94,0.97,0.16,0.04,1.00,-0.03,-0.03,-0.03,-0.03,0.70,-0.04,0.72,0.67,0.16,NaN,NaN
